# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassaanSaqib/FlyRankAI-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — “Which Pages Will Grow?”

The paper reports that its growth-prediction model achieved approximately **90% accuracy on unseen pages belonging to brands represented during training**, compared with approximately **75% on brands that were completely unseen during training**.

My methodology questions are:

1. How exactly was the growth-versus-decline label constructed, including its measurement window and thresholds?

2. Were all predictive features calculated strictly before the outcome window?

3. For the unseen-brand evaluation, were all pages belonging to each brand kept entirely within one fold, with no brand appearing in both training and testing?

4. Were model selection and preprocessing performed only on training data before the held-out brands were evaluated?

These questions are constructive because the reported difference between known and unseen brands is useful. Clarifying the label timing and grouping design would show whether the evidence supports within-brand ranking, cross-brand generalisation, or both.

### Finding 2 — “Refreshing Pages Actually Works”

The paper reports that **7 of 9 analysed strata showed statistically significant refresh lift**.

My methodology questions are:

1. How was a refresh defined and timestamped?

2. What outcome window was measured after the refresh?

3. How were refreshed pages compared with similar pages that were not refreshed?

4. Were prior momentum, visibility, page age, competition and client-level differences controlled through matching, stratification or another design?

5. Could pages selected for refresh already have been more visible or more promising than pages that were left unchanged?

This is not a criticism of the analysis. The observed comparison can provide valuable directional evidence and decision support. However, language such as “refreshing works” may sound causal, so I would want to understand how the validation design separates refresh effects from page-selection effects.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

import subprocess

import numpy as np

import pandas as pd

import sklearn

from IPython.display import display, Markdown

from sklearn.model_selection import (

    train_test_split,

    GroupShuffleSplit,

)

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (

    average_precision_score,

    roc_auc_score,

)

RANDOM_STATE = 42

paper_findings = pd.DataFrame(

    [

        {

            "Paper finding": "Which Pages Will Grow?",

            "Reported result": (

                "Approximately 90% accuracy on unseen pages from "

                "known brands and 75% on completely unseen brands."

            ),

            "Methodology question": (

                "How was the growth/decline label created, were all "

                "features fixed before the outcome window, and were "

                "brands kept completely separate during grouped validation?"

            ),

            "Why it matters": (

                "This determines whether the result supports "

                "within-brand ranking, cross-brand generalisation, or both."

            ),

        },

        {

            "Paper finding": "Refreshing Pages Actually Works",

            "Reported result": (

                "Seven of nine analysed strata showed statistically "

                "significant refresh lift."

            ),

            "Methodology question": (

                "How was refresh timing defined, what post-refresh "

                "window was measured, and were refreshed pages matched "

                "with comparable unrefreshed pages?"

            ),

            "Why it matters": (

                "An observed difference supports directional decisions, "

                "while a causal claim requires stronger control of "

                "page-selection effects."

            ),

        },

    ]

)

display(Markdown("### Constructive paper methodology audit"))

display(paper_findings)

print("Two paper findings documented.")

print("Each finding includes a label or validation-design question.")

### Constructive paper methodology audit

,Paper finding,Reported result,Methodology question,Why it matters
0,Which Pages Will Grow?,Approximately 90% accuracy on unseen pages fro...,"How was the growth/decline label created, were...",This determines whether the result supports wi...
1,Refreshing Pages Actually Works,Seven of nine analysed strata showed statistic...,"How was refresh timing defined, what post-refr...",An observed difference supports directional de...


Two paper findings documented.
Each finding includes a label or validation-design question.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 analysis already used a grouped client holdout as its main validation design. To demonstrate why that choice matters, I compare it with a less reliable **random row split**.

In the before condition, rows are randomly divided while preserving the overall label ratio. Because one client has many pages, pages from the same client can appear in both training and testing. This may allow the model to benefit from repeated client-specific patterns.

In the improved condition, I use a **grouped client holdout**. Approximately 80% of the pseudonymized clients are used for training and the remaining clients are held out for testing. Client overlap must equal zero.

I re-run the same Week-5 candidate methods:

- Logistic Regression

- Random Forest

The models use the same features and fixed random state as Week 5. I report the observed base rate beside Precision@20, Precision@50, average precision and ROC AUC. The grouped result is treated as the more defensible estimate because it asks whether the model transfers to clients it did not see during training.

I also inspect three public-safe failure examples from the selected grouped model. The 0.50 threshold is used only to label false positives and false negatives for inspection; the main capstone use remains ranked decision support.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------------

# 2A. Load the anonymized starter data

# ------------------------------------------------------------

REPO_URL = (

    "https://github.com/HassaanSaqib/"

    "FlyRankAI-Internship.git"

)

REPO_DIR = Path("/content/flyrank_w06_repo")

DATA_PATH = (

    REPO_DIR

    / "data"

    / "raw"

    / "content_refresh_anonymized.csv"

)

if not REPO_DIR.exists():

    subprocess.run(

        [

            "git",

            "clone",

            "-q",

            REPO_URL,

            str(REPO_DIR),

        ],

        check=True,

    )

if not DATA_PATH.exists():

    raise FileNotFoundError(

        f"Dataset was not found at {DATA_PATH}"

    )

df = pd.read_csv(DATA_PATH)

required_columns = {

    "content_id",

    "client_id",

    "trend_direction",

    "trend_pct",

    "impressions_90d",

    "clicks_90d",

    "sessions_90d",

    "ctr",

    "avg_position",

    "days_since_last_update",

    "content_age_days",

    "days_with_impressions",

    "engagement_rate",

    "scroll_rate",

    "word_count",

    "impressions_prev_30d",

    "clicks_prev_30d",

    "sessions_prev_30d",

}

missing_columns = required_columns.difference(

    df.columns

)

if missing_columns:

    raise KeyError(

        "Required columns are missing: "

        f"{sorted(missing_columns)}"

    )

numeric_columns = sorted(

    required_columns

    - {

        "content_id",

        "client_id",

        "trend_direction",

    }

)

for column in numeric_columns:

    df[column] = pd.to_numeric(

        df[column],

        errors="coerce",

    )

# Retrospective evaluation outcome.

df["is_declining_eval"] = (

    df["trend_direction"]

    .astype(str)

    .str.lower()

    .eq("down")

    .astype(int)

)

# ------------------------------------------------------------

# 2B. Recreate the Week-5 feature set

# ------------------------------------------------------------

df["log_impressions_90d"] = np.log1p(

    df["impressions_90d"].clip(lower=0)

)

df["log_clicks_90d"] = np.log1p(

    df["clicks_90d"].clip(lower=0)

)

df["log_sessions_90d"] = np.log1p(

    df["sessions_90d"].clip(lower=0)

)

# Average position zero means no measured position.

df["avg_position_clean"] = (

    df["avg_position"].where(

        df["avg_position"] > 0,

        np.nan,

    )

)

df["has_position"] = (

    df["avg_position_clean"]

    .notna()

    .astype(int)

)

df["has_word_count"] = (

    df["word_count"]

    .notna()

    .astype(int)

)

df["has_engagement_rate"] = (

    df["engagement_rate"]

    .notna()

    .astype(int)

)

df["has_scroll_rate"] = (

    df["scroll_rate"]

    .notna()

    .astype(int)

)

df["has_days_since_update"] = (

    df["days_since_last_update"]

    .notna()

    .astype(int)

)

feature_columns = [

    "log_impressions_90d",

    "log_clicks_90d",

    "log_sessions_90d",

    "ctr",

    "avg_position_clean",

    "days_since_last_update",

    "content_age_days",

    "days_with_impressions",

    "engagement_rate",

    "scroll_rate",

    "word_count",

    "has_position",

    "has_word_count",

    "has_engagement_rate",

    "has_scroll_rate",

    "has_days_since_update",

]

X = df[feature_columns].copy()

y = df["is_declining_eval"].astype(int)

groups = (

    df["client_id"]

    .fillna("unknown")

    .astype(str)

)

# ------------------------------------------------------------

# 2C. Model and metric helpers

# ------------------------------------------------------------

def build_model(model_name):

    """Return a fresh copy of a Week-5 candidate model."""

    if model_name == "Logistic Regression":

        return Pipeline(

            steps=[

                (

                    "imputer",

                    SimpleImputer(

                        strategy="median"

                    ),

                ),

                (

                    "scaler",

                    StandardScaler(),

                ),

                (

                    "model",

                    LogisticRegression(

                        class_weight="balanced",

                        max_iter=2000,

                        random_state=RANDOM_STATE,

                    ),

                ),

            ]

        )

    if model_name == "Random Forest":

        return Pipeline(

            steps=[

                (

                    "imputer",

                    SimpleImputer(

                        strategy="median"

                    ),

                ),

                (

                    "model",

                    RandomForestClassifier(

                        n_estimators=300,

                        max_depth=8,

                        min_samples_leaf=25,

                        class_weight=(

                            "balanced_subsample"

                        ),

                        random_state=RANDOM_STATE,

                        n_jobs=-1,

                    ),

                ),

            ]

        )

    raise ValueError(

        f"Unknown model: {model_name}"

    )

def precision_at_k(y_true, scores, k):

    """Observed positive rate among the k highest scores."""

    ranked = pd.DataFrame(

        {

            "target": np.asarray(

                y_true,

                dtype=int,

            ),

            "score": np.asarray(

                scores,

                dtype=float,

            ),

        }

    )

    top_k = (

        ranked.sort_values(

            "score",

            ascending=False,

            kind="mergesort",

        )

        .head(min(k, len(ranked)))

    )

    if top_k.empty:

        return 0.0

    return float(

        top_k["target"].mean()

    )

def evaluate_ranking(y_true, scores):

    """Use the same metrics for every split and model."""

    y_array = np.asarray(

        y_true,

        dtype=int,

    )

    score_array = np.asarray(

        scores,

        dtype=float,

    )

    return {

        "Base rate": float(

            y_array.mean()

        ),

        "Precision@20": precision_at_k(

            y_array,

            score_array,

            20,

        ),

        "Precision@50": precision_at_k(

            y_array,

            score_array,

            50,

        ),

        "Average precision": float(

            average_precision_score(

                y_array,

                score_array,

            )

        ),

        "ROC AUC": float(

            roc_auc_score(

                y_array,

                score_array,

            )

        ),

    }

# ------------------------------------------------------------

# 2D. Before: random row split

# ------------------------------------------------------------

all_indices = np.arange(

    len(df)

)

row_train_idx, row_test_idx = (

    train_test_split(

        all_indices,

        test_size=0.20,

        stratify=y,

        random_state=RANDOM_STATE,

    )

)

# ------------------------------------------------------------

# 2E. After: grouped client holdout

# ------------------------------------------------------------

group_splitter = GroupShuffleSplit(

    n_splits=20,

    test_size=0.20,

    random_state=RANDOM_STATE,

)

grouped_indices = None

for candidate_train, candidate_test in (

    group_splitter.split(

        X,

        y,

        groups,

    )

):

    train_class_count = (

        y.iloc[candidate_train]

        .nunique()

    )

    test_class_count = (

        y.iloc[candidate_test]

        .nunique()

    )

    if (

        train_class_count == 2

        and test_class_count == 2

    ):

        grouped_indices = (

            np.asarray(candidate_train),

            np.asarray(candidate_test),

        )

        break

if grouped_indices is None:

    raise ValueError(

        "A valid grouped split could not be created."

    )

group_train_idx, group_test_idx = (

    grouped_indices

)

splits = {

    "Random row split (before)": (

        row_train_idx,

        row_test_idx,

    ),

    "Grouped client holdout (after)": (

        group_train_idx,

        group_test_idx,

    ),

}

candidate_model_names = [

    "Logistic Regression",

    "Random Forest",

]

audit_rows = []

fitted_models = {}

score_store = {}

# ------------------------------------------------------------

# 2F. Train every method under both designs

# ------------------------------------------------------------

for split_name, indices in splits.items():

    train_idx, test_idx = indices

    train_clients = set(

        groups.iloc[train_idx]

    )

    test_clients = set(

        groups.iloc[test_idx]

    )

    overlap_count = len(

        train_clients.intersection(

            test_clients

        )

    )

    for model_name in candidate_model_names:

        model = build_model(

            model_name

        )

        model.fit(

            X.iloc[train_idx],

            y.iloc[train_idx],

        )

        scores = model.predict_proba(

            X.iloc[test_idx]

        )[:, 1]

        metrics = evaluate_ranking(

            y.iloc[test_idx],

            scores,

        )

        audit_rows.append(

            {

                "Split": split_name,

                "Model": model_name,

                "Train rows": len(train_idx),

                "Test rows": len(test_idx),

                "Train clients": len(

                    train_clients

                ),

                "Test clients": len(

                    test_clients

                ),

                "Client overlap": (

                    overlap_count

                ),

                **metrics,

            }

        )

        fitted_models[

            (split_name, model_name)

        ] = model

        score_store[

            (split_name, model_name)

        ] = scores

split_audit_table = pd.DataFrame(

    audit_rows

)

display(

    Markdown(

        "### Before-and-after validation comparison"

    )

)

display(

    split_audit_table.style.format(

        {

            "Train rows": "{:,.0f}",

            "Test rows": "{:,.0f}",

            "Train clients": "{:,.0f}",

            "Test clients": "{:,.0f}",

            "Client overlap": "{:,.0f}",

            "Base rate": "{:.1%}",

            "Precision@20": "{:.1%}",

            "Precision@50": "{:.1%}",

            "Average precision": "{:.3f}",

            "ROC AUC": "{:.3f}",

        }

    )

)

# ------------------------------------------------------------

# 2G. Select the learned method using grouped performance

# ------------------------------------------------------------

grouped_results = (

    split_audit_table[

        split_audit_table["Split"].eq(

            "Grouped client holdout (after)"

        )

    ]

    .sort_values(

        [

            "Precision@20",

            "Average precision",

        ],

        ascending=False,

    )

    .reset_index(drop=True)

)

selected_model_name = (

    grouped_results.iloc[0]["Model"]

)

selected_before_after = (

    split_audit_table[

        split_audit_table["Model"].eq(

            selected_model_name

        )

    ]

    .copy()

)

display(

    Markdown(

        "### Selected Week-5 model: "

        f"{selected_model_name}"

    )

)

display(

    selected_before_after[

        [

            "Split",

            "Model",

            "Client overlap",

            "Base rate",

            "Precision@20",

            "Precision@50",

            "Average precision",

            "ROC AUC",

        ]

    ].style.format(

        {

            "Client overlap": "{:,.0f}",

            "Base rate": "{:.1%}",

            "Precision@20": "{:.1%}",

            "Precision@50": "{:.1%}",

            "Average precision": "{:.3f}",

            "ROC AUC": "{:.3f}",

        }

    )

)

selected_grouped_model = fitted_models[

    (

        "Grouped client holdout (after)",

        selected_model_name,

    )

]

selected_grouped_scores = score_store[

    (

        "Grouped client holdout (after)",

        selected_model_name,

    )

]

random_selected_row = (

    selected_before_after[

        selected_before_after["Split"].eq(

            "Random row split (before)"

        )

    ]

    .iloc[0]

)

grouped_selected_row = (

    selected_before_after[

        selected_before_after["Split"].eq(

            "Grouped client holdout (after)"

        )

    ]

    .iloc[0]

)

ap_change = (

    grouped_selected_row[

        "Average precision"

    ]

    - random_selected_row[

        "Average precision"

    ]

)

if ap_change < 0:

    change_text = (

        f"decreased by {abs(ap_change):.3f}"

    )

elif ap_change > 0:

    change_text = (

        f"increased by {ap_change:.3f}"

    )

else:

    change_text = "did not change"

display(

    Markdown(

        f"""

**Observed validation result:** The random split contained

**{int(random_selected_row['Client overlap'])} overlapping clients**,

while the grouped holdout contained **0**.

For the selected method, average precision {change_text} after

switching to the grouped client holdout. The grouped estimate is the

more defensible result because it measures performance on clients the

model did not observe during training.

"""

    )

)

# ------------------------------------------------------------

# 2H. Inspect three real grouped-holdout errors

# ------------------------------------------------------------

group_test_frame = (

    df.iloc[group_test_idx]

    .copy()

)

group_test_y = (

    y.iloc[group_test_idx]

    .to_numpy()

)

diagnostic_predictions = (

    selected_grouped_scores >= 0.50

).astype(int)

error_frame = group_test_frame[

    [

        "impressions_90d",

        "ctr",

        "avg_position",

        "days_since_last_update",

        "content_age_days",

    ]

].copy()

error_frame["observed_label"] = (

    group_test_y

)

error_frame["model_score"] = (

    selected_grouped_scores

)

error_frame["predicted_label"] = (

    diagnostic_predictions

)

error_frame["error_type"] = np.select(

    [

        (

            error_frame[

                "observed_label"

            ].eq(0)

            & error_frame[

                "predicted_label"

            ].eq(1)

        ),

        (

            error_frame[

                "observed_label"

            ].eq(1)

            & error_frame[

                "predicted_label"

            ].eq(0)

        ),

    ],

    [

        "false_positive",

        "false_negative",

    ],

    default="correct",

)

false_positives = (

    error_frame[

        error_frame["error_type"].eq(

            "false_positive"

        )

    ]

    .sort_values(

        "model_score",

        ascending=False,

    )

    .head(2)

)

false_negatives = (

    error_frame[

        error_frame["error_type"].eq(

            "false_negative"

        )

    ]

    .sort_values(

        "model_score",

        ascending=True,

    )

    .head(2)

)

wrong_cases = pd.concat(

    [

        false_positives,

        false_negatives,

    ]

).head(3).copy()

wrong_cases.insert(

    0,

    "case",

    [

        f"error_case_{number}"

        for number in range(

            1,

            len(wrong_cases) + 1,

        )

    ],

)

display(

    Markdown(

        "### Three public-safe failure examples"

    )

)

display(

    wrong_cases.style.format(

        {

            "impressions_90d": "{:,.0f}",

            "ctr": "{:.2f}%",

            "avg_position": "{:.1f}",

            "days_since_last_update": "{:.0f}",

            "content_age_days": "{:.0f}",

            "model_score": "{:.3f}",

        }

    )

)

error_counts = (

    error_frame["error_type"]

    .value_counts()

    .rename_axis("Error type")

    .reset_index(name="Rows")

)

display(

    Markdown(

        "### Diagnostic error counts at threshold 0.50"

    )

)

display(error_counts)

### Before-and-after validation comparison

,Split,Model,Train rows,Test rows,Train clients,Test clients,Client overlap,Base rate,Precision@20,Precision@50,Average precision,ROC AUC
0,Random row split (before),Logistic Regression,"24,000","6,000",32,31,31,54.2%,90.0%,88.0%,0.717,0.701
1,Random row split (before),Random Forest,"24,000","6,000",32,31,31,54.2%,95.0%,94.0%,0.760,0.751
2,Grouped client holdout (after),Logistic Regression,"23,837","6,163",25,7,0,51.1%,80.0%,84.0%,0.632,0.630
3,Grouped client holdout (after),Random Forest,"23,837","6,163",25,7,0,51.1%,45.0%,62.0%,0.602,0.614


### Selected Week-5 model: Logistic Regression

,Split,Model,Client overlap,Base rate,Precision@20,Precision@50,Average precision,ROC AUC
0,Random row split (before),Logistic Regression,31,54.2%,90.0%,88.0%,0.717,0.701
2,Grouped client holdout (after),Logistic Regression,0,51.1%,80.0%,84.0%,0.632,0.630




**Observed validation result:** The random split contained

**31 overlapping clients**,

while the grouped holdout contained **0**.

For the selected method, average precision decreased by 0.085 after

switching to the grouped client holdout. The grouped estimate is the

more defensible result because it measures performance on clients the

model did not observe during training.



### Three public-safe failure examples

,case,impressions_90d,ctr,avg_position,days_since_last_update,content_age_days,observed_label,model_score,predicted_label,error_type
27993,error_case_1,"1,266",0.00%,4.6,106,106,0,0.903,1,false_positive
12869,error_case_2,"15,101",0.00%,5.7,7,421,0,0.886,1,false_positive
27271,error_case_3,1,0.00%,0.0,92,238,1,0.009,0,false_negative


### Diagnostic error counts at threshold 0.50

,Error type,Rows
0,correct,3672
1,false_positive,1374
2,false_negative,1117


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audit three possible leakage routes.

### 1. Direct label and identifier leakage

The final feature list must not contain:

- `trend_direction`

- `trend_pct`

- `is_declining_eval`

- `is_declining_label`

- `content_id`

- `client_id`

Client ID is used only to construct the grouped validation split.

### 2. Decision-derived leakage

The model must not use the Week-4 baseline score, stale flag, low-CTR flag, reason code or action label. Those fields belong to the comparison baseline and recommendation layer, not to the learned feature set.

### 3. Overlapping-window leakage

This audit identified the most important remaining limitation. The retrospective decline label compares impressions in the latest 30 days with impressions in the preceding 30 days. Several Week-5 inputs are trailing-90-day totals or rates. Those measurements contain the same latest-30-day period used by the outcome.

Therefore, the original grouped Week-5 score is useful as an **observed retrospective association**, but it is not a clean forward-prediction estimate.

To test this honestly, I run two additional grouped-client models:

- a **strict pre-outcome model**, using only measurements from the previous 30-day window;

- a deliberately invalid model containing `trend_pct`, used only as a leakage probe.

If adding `trend_pct` makes the score nearly perfect, the test harness is correctly detecting an answer-derived feature. That probe is then removed and is never treated as a valid result

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------------

# 3A. Direct label, identifier and decision-feature audit

# ------------------------------------------------------------

exact_forbidden_features = {

    "content_id",

    "client_id",

    "trend_direction",

    "trend_pct",

    "is_declining_eval",

    "is_declining_label",

}

decision_derived_features = {

    "baseline_score",

    "raw_score",

    "stale_flag",

    "low_ctr_flag",

    "action_label",

    "reason_code",

}

direct_leaks = sorted(

    set(feature_columns).intersection(

        exact_forbidden_features

        | decision_derived_features

    )

)

direct_audit = pd.DataFrame(

    {

        "Audit": [

            "Label-derived columns in features",

            "Identifiers in features",

            "Decision-derived fields in features",

            "client_id used as model feature",

            "client_id used for grouping",

        ],

        "Result": [

            (

                "PASS"

                if not set(feature_columns).intersection(

                    {

                        "trend_direction",

                        "trend_pct",

                        "is_declining_eval",

                        "is_declining_label",

                    }

                )

                else "FAIL"

            ),

            (

                "PASS"

                if not set(feature_columns).intersection(

                    {

                        "content_id",

                        "client_id",

                    }

                )

                else "FAIL"

            ),

            (

                "PASS"

                if not set(feature_columns).intersection(

                    decision_derived_features

                )

                else "FAIL"

            ),

            (

                "No"

                if "client_id"

                not in feature_columns

                else "Yes"

            ),

            "Yes",

        ],

    }

)

display(

    Markdown(

        "### Direct leakage audit"

    )

)

display(direct_audit)

assert not direct_leaks, (

    "Forbidden features detected: "

    f"{direct_leaks}"

)

# ------------------------------------------------------------

# 3B. Audit the timing and lineage of every Week-5 feature

# ------------------------------------------------------------

overlapping_window_features = {

    "log_impressions_90d",

    "log_clicks_90d",

    "log_sessions_90d",

    "ctr",

    "avg_position_clean",

    "days_with_impressions",

    "engagement_rate",

    "scroll_rate",

    "has_position",

    "has_engagement_rate",

    "has_scroll_rate",

}

snapshot_review_features = {

    "days_since_last_update",

    "word_count",

    "has_word_count",

    "has_days_since_update",

}

reconstructable_features = {

    "content_age_days",

}

def feature_timing_risk(feature):

    """Describe the timing risk of a Week-5 feature."""

    if feature in overlapping_window_features:

        return (

            "High",

            (

                "The trailing-90-day measurement "

                "contains the latest-30-day outcome window."

            ),

        )

    if feature in snapshot_review_features:

        return (

            "Review",

            (

                "Measured at snapshot end and may reflect "

                "an edit or update occurring during the "

                "outcome period."

            ),

        )

    if feature in reconstructable_features:

        return (

            "Low",

            (

                "Age can be shifted back by 30 days to "

                "represent prediction-time age; the constant "

                "shift does not add outcome information."

            ),

        )

    return (

        "Review",

        "Timing and lineage require manual confirmation.",

    )

feature_audit_rows = []

for feature in feature_columns:

    risk_level, reason = (

        feature_timing_risk(feature)

    )

    feature_audit_rows.append(

        {

            "Feature": feature,

            "Timing risk": risk_level,

            "Reason": reason,

        }

    )

feature_audit_table = pd.DataFrame(

    feature_audit_rows

)

display(

    Markdown(

        "### Final feature timing audit"

    )

)

display(feature_audit_table)

high_risk_count = int(

    feature_audit_table[

        "Timing risk"

    ].eq("High").sum()

)

# ------------------------------------------------------------

# 3C. Build a strict pre-outcome feature set

# ------------------------------------------------------------

# Previous-window features are known before the latest

# 30-day outcome period begins.

df["log_impressions_prev_30d"] = np.log1p(

    df["impressions_prev_30d"].clip(

        lower=0

    )

)

df["log_clicks_prev_30d"] = np.log1p(

    df["clicks_prev_30d"].clip(

        lower=0

    )

)

df["log_sessions_prev_30d"] = np.log1p(

    df["sessions_prev_30d"].clip(

        lower=0

    )

)

df["ctr_prev_30d"] = np.where(

    df["impressions_prev_30d"] > 0,

    (

        100

        * df["clicks_prev_30d"]

        / df["impressions_prev_30d"]

    ),

    np.nan,

)

df[

    "sessions_per_100_impressions_prev_30d"

] = np.where(

    df["impressions_prev_30d"] > 0,

    (

        100

        * df["sessions_prev_30d"]

        / df["impressions_prev_30d"]

    ),

    np.nan,

)

df["has_prev_impressions"] = (

    df["impressions_prev_30d"]

    .gt(0)

    .astype(int)

)

df["has_prev_clicks"] = (

    df["clicks_prev_30d"]

    .gt(0)

    .astype(int)

)

df["has_prev_sessions"] = (

    df["sessions_prev_30d"]

    .gt(0)

    .astype(int)

)

safe_feature_columns = [

    "log_impressions_prev_30d",

    "log_clicks_prev_30d",

    "log_sessions_prev_30d",

    "ctr_prev_30d",

    (

        "sessions_per_100_"

        "impressions_prev_30d"

    ),

    "has_prev_impressions",

    "has_prev_clicks",

    "has_prev_sessions",

]

safe_forbidden_overlap = sorted(

    set(safe_feature_columns).intersection(

        exact_forbidden_features

        | decision_derived_features

    )

)

assert not safe_forbidden_overlap, (

    "Forbidden fields entered the strict feature set: "

    f"{safe_forbidden_overlap}"

)

timeline_table = pd.DataFrame(

    {

        "Stage": [

            "Feature window",

            "Prediction point",

            "Outcome window",

            "Evaluation split",

        ],

        "Design": [

            (

                "Previous 30 days: days 31–60 "

                "before snapshot end"

            ),

            (

                "Immediately before the latest "

                "30-day period"

            ),

            (

                "Latest 30 days, represented by "

                "the retrospective decline label"

            ),

            (

                "Held-out clients with zero "

                "client overlap"

            ),

        ],

    }

)

display(

    Markdown(

        "### Strict feature/label timeline"

    )

)

display(timeline_table)

# ------------------------------------------------------------

# 3D. Re-run selected model with strict features

# ------------------------------------------------------------

strict_model = build_model(

    selected_model_name

)

strict_model.fit(

    df.iloc[group_train_idx][

        safe_feature_columns

    ],

    y.iloc[group_train_idx],

)

strict_scores = strict_model.predict_proba(

    df.iloc[group_test_idx][

        safe_feature_columns

    ]

)[:, 1]

# ------------------------------------------------------------

# 3E. Deliberate leakage probe

# ------------------------------------------------------------

# trend_pct directly creates trend_direction and is therefore

# invalid. It is added once only to verify the audit harness.

# It is not a valid model or capstone result.

leaky_probe_columns = (

    safe_feature_columns

    + ["trend_pct"]

)

leaky_probe_model = build_model(

    selected_model_name

)

leaky_probe_model.fit(

    df.iloc[group_train_idx][

        leaky_probe_columns

    ],

    y.iloc[group_train_idx],

)

leaky_probe_scores = (

    leaky_probe_model.predict_proba(

        df.iloc[group_test_idx][

            leaky_probe_columns

        ]

    )[:, 1]

)

# ------------------------------------------------------------

# 3F. Compare feature designs on the same grouped holdout

# ------------------------------------------------------------

leakage_comparison_rows = []

feature_designs = [

    (

        "Week-5 feature set",

        selected_grouped_scores,

        "Retrospective association only",

    ),

    (

        "Strict pre-outcome feature set",

        strict_scores,

        "Yes — more defensible",

    ),

    (

        "Deliberate trend_pct leakage probe",

        leaky_probe_scores,

        "No — diagnostic only",

    ),

]

for design_name, scores, validity in (

    feature_designs

):

    metrics = evaluate_ranking(

        y.iloc[group_test_idx],

        scores,

    )

    leakage_comparison_rows.append(

        {

            "Feature design": design_name,

            "Valid for forward-looking claim?": (

                validity

            ),

            **metrics,

        }

    )

leakage_comparison = pd.DataFrame(

    leakage_comparison_rows

)

display(

    Markdown(

        "### Leakage sensitivity comparison"

    )

)

display(

    leakage_comparison.style.format(

        {

            "Base rate": "{:.1%}",

            "Precision@20": "{:.1%}",

            "Precision@50": "{:.1%}",

            "Average precision": "{:.3f}",

            "ROC AUC": "{:.3f}",

        }

    )

)

strict_result = (

    leakage_comparison[

        leakage_comparison[

            "Feature design"

        ].eq(

            "Strict pre-outcome feature set"

        )

    ]

    .iloc[0]

)

leaky_result = (

    leakage_comparison[

        leakage_comparison[

            "Feature design"

        ].eq(

            "Deliberate trend_pct leakage probe"

        )

    ]

    .iloc[0]

)

display(

    Markdown(

        f"""

### Leakage-audit interpretation

The exact-name audit passed: no identifiers, label-derived columns or

Week-4 decision flags were present in the Week-5 feature list.

However, **{high_risk_count} Week-5 features** were marked as carrying

overlapping-window risk. Therefore, the original Week-5 grouped score

is framed as an observed retrospective association rather than a

clean future prediction.

After restricting the model to measurements available before the

latest 30-day outcome period, the selected model achieved:

- **Precision@20:** {strict_result['Precision@20']:.1%}

- **Average precision:** {strict_result['Average precision']:.3f}

- **ROC AUC:** {strict_result['ROC AUC']:.3f}

- **Held-out base rate:** {strict_result['Base rate']:.1%}

The deliberately invalid `trend_pct` probe produced average precision

of **{leaky_result['Average precision']:.3f}**. Its near-perfect result

confirms that the audit harness reacts strongly when the answer itself

is inserted as a feature. That column is removed and is not used for

any valid claim or recommendation.

"""

    )

)

### Direct leakage audit

,Audit,Result
0,Label-derived columns in features,PASS
1,Identifiers in features,PASS
2,Decision-derived fields in features,PASS
3,client_id used as model feature,No
4,client_id used for grouping,Yes


### Final feature timing audit

,Feature,Timing risk,Reason
0,log_impressions_90d,High,The trailing-90-day measurement contains the l...
1,log_clicks_90d,High,The trailing-90-day measurement contains the l...
2,log_sessions_90d,High,The trailing-90-day measurement contains the l...
3,ctr,High,The trailing-90-day measurement contains the l...
4,avg_position_clean,High,The trailing-90-day measurement contains the l...
5,days_since_last_update,Review,Measured at snapshot end and may reflect an ed...
6,content_age_days,Low,Age can be shifted back by 30 days to represen...
7,days_with_impressions,High,The trailing-90-day measurement contains the l...
8,engagement_rate,High,The trailing-90-day measurement contains the l...
9,scroll_rate,High,The trailing-90-day measurement contains the l...


### Strict feature/label timeline

,Stage,Design
0,Feature window,Previous 30 days: days 31–60 before snapshot end
1,Prediction point,Immediately before the latest 30-day period
2,Outcome window,"Latest 30 days, represented by the retrospecti..."
3,Evaluation split,Held-out clients with zero client overlap


### Leakage sensitivity comparison

,Feature design,Valid for forward-looking claim?,Base rate,Precision@20,Precision@50,Average precision,ROC AUC
0,Week-5 feature set,Retrospective association only,51.1%,80.0%,84.0%,0.632,0.630
1,Strict pre-outcome feature set,Yes — more defensible,51.1%,65.0%,64.0%,0.594,0.626
2,Deliberate trend_pct leakage probe,No — diagnostic only,51.1%,100.0%,100.0%,1.000,1.000




### Leakage-audit interpretation

The exact-name audit passed: no identifiers, label-derived columns or

Week-4 decision flags were present in the Week-5 feature list.

However, **11 Week-5 features** were marked as carrying

overlapping-window risk. Therefore, the original Week-5 grouped score

is framed as an observed retrospective association rather than a

clean future prediction.

After restricting the model to measurements available before the

latest 30-day outcome period, the selected model achieved:

- **Precision@20:** 65.0%

- **Average precision:** 0.594

- **ROC AUC:** 0.626

- **Held-out base rate:** 51.1%

The deliberately invalid `trend_pct` probe produced average precision

of **1.000**. Its near-perfect result

confirms that the audit harness reacts strongly when the answer itself

is inserted as a feature. That column is removed and is not used for

any valid claim or recommendation.



## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Overstated version

> The model predicts which pages need refreshing and proves that traffic, freshness and engagement features drive content decline.

### Evidence-aligned rewrite

On this anonymized starter dataset, the model measured directional associations between available page-level signals and a retrospective decline label. Performance was lower under a client-grouped holdout than under a random row split, showing that a random split gave a more optimistic estimate when pages from the same clients appeared in both sets.

The leakage audit also found that several Week-5 trailing-90-day features overlapped the latest-30-day outcome period. I therefore do not present the Week-5 result as clean future prediction. A stricter model using only previous-window measurements provides a more defensible forward-looking estimate, although its performance remains limited to this dataset, this label definition and 32 pseudonymized clients.

The resulting score is **directional decision support** for ranking pages for human review. It does not prove Google's ranking algorithm, establish that any feature causes decline, or demonstrate that refreshing a selected page will causally improve future search performance.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grouped_metric_row = (

    selected_before_after[

        selected_before_after[

            "Split"

        ].eq(

            "Grouped client holdout (after)"

        )

    ]

    .iloc[0]

)

strict_metric_row = (

    leakage_comparison[

        leakage_comparison[

            "Feature design"

        ].eq(

            "Strict pre-outcome feature set"

        )

    ]

    .iloc[0]

)

overstated_claim = (

    "The model predicts which pages need refreshing "

    "and proves that traffic, freshness and engagement "

    "features drive content decline."

)

safe_claim = (

    "On the anonymized starter dataset, the selected "

    f"{selected_model_name} model measured directional "

    "associations with a retrospective decline label. "

    "Under the grouped client holdout, its observed "

    f"Precision@20 was "

    f"{grouped_metric_row['Precision@20']:.1%}, compared "

    f"with a held-out base rate of "

    f"{grouped_metric_row['Base rate']:.1%}. "

    "Because several Week-5 trailing-90-day measurements "

    "overlapped the outcome period, that result is not "

    "presented as clean future prediction. Using only "

    "strict pre-outcome measurements, observed "

    f"Precision@20 was "

    f"{strict_metric_row['Precision@20']:.1%} and average "

    f"precision was "

    f"{strict_metric_row['Average precision']:.3f}. "

    "These measurements provide directional "

    "decision-support for human page review. They do not "

    "prove Google's algorithm, establish causal feature "

    "effects, or prove that refreshing a selected page "

    "will improve future performance."

)

claim_rewrite_table = pd.DataFrame(

    {

        "Version": [

            "Overstated",

            "Evidence-aligned",

        ],

        "Claim": [

            overstated_claim,

            safe_claim,

        ],

    }

)

display(

    Markdown(

        "### Before-and-after claim rewrite"

    )

)

display(claim_rewrite_table)

required_safe_terms = [

    "observed",

    "measured",

    "directional",

    "decision-support",

]

language_check = pd.DataFrame(

    {

        "Required safe term": (

            required_safe_terms

        ),

        "Present in rewritten claim": [

            term.lower()

            in safe_claim.lower()

            for term in required_safe_terms

        ],

    }

)

display(

    Markdown(

        "### Safe-language check"

    )

)

display(language_check)

assert language_check[

    "Present in rewritten claim"

].all(), (

    "The rewritten claim is missing one or more "

    "required safe-language terms."

)

print("Claim rewrite complete.")

print("Safe-language check: PASS")

### Before-and-after claim rewrite

,Version,Claim
0,Overstated,The model predicts which pages need refreshing...
1,Evidence-aligned,"On the anonymized starter dataset, the selecte..."


### Safe-language check

,Required safe term,Present in rewritten claim
0,observed,True
1,measured,True
2,directional,True
3,decision-support,True


Claim rewrite complete.
Safe-language check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.